In [70]:
import sys
import torch

import numpy as np
import trimesh
import plotly.graph_objects as go

In [2]:
sys.path.append("..")

In [3]:
from utils.grasp_utils import get_handmodel

In [4]:
from model.hand_opt import AdamGraspTransfer

## Info

In [5]:
source_gripper = "mano_right"
target_gripper = "fetch_gripper"
device = "cpu"

## Hand Models

In [6]:
source_model = get_handmodel(
  source_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

In [7]:
target_model = get_handmodel(
  target_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

In [ ]:
target_model.dynamic_joints_q_upper

## Source Gripper Pose

In [ ]:
grasp_pose = torch.zeros(9)
grasp_pose[0:3] = torch.tensor([0.1, 0.2, 0.3])
# Identity rotation in 6d rot representation is: (1,0,0,0,1,0)
grasp_pose[3] = 1
grasp_pose[7] = 1 
print("Pose:", grasp_pose)


grasp_dofs_lower = source_model.dynamic_joints_q_lower.squeeze(0).clone()
grasp_dofs_mid = torch.tensor(source_model.dynamic_joints_q_mid)

grasp_dofs = grasp_dofs_lower + grasp_dofs_mid
# grasp_dofs = grasp_dofs_mid # for shadowhand
# scale = 0.5
# grasp_dofs = scale * torch.rand_like(grasp_dofs_mid) * grasp_dofs_mid + grasp_dofs_lower



print("Dofs:", grasp_dofs)

sample_grasp_q = (
  torch.cat(
    [
      grasp_pose,
      grasp_dofs,
    ]
  )
  .unsqueeze(0)
  .to(device)
  .float()
)

In [ ]:
sample_grasp_q.shape

In [11]:
# print("Plotting SOURCE...")
# vis_data = source_model.get_plotly_data(q=sample_grasp_q)
# fig = go.Figure(data=vis_data)
# # fig.update_layout(template='simple_white')
# # fig.update_xaxes(showgrid=False)
# # fig.update_yaxes(showgrid=False)
# fig.update_layout(
#     scene = dict(
#         xaxis = dict(visible=False),
#         yaxis = dict(visible=False),
#         zaxis =dict(visible=False)
#         )
# )
# fig.show()


## Grasp Transfer Optimization

In [39]:
grasp_transfer_opt = AdamGraspTransfer(
  source_gripper,
  target_gripper,
  learning_rate=1e-3,
  device=device
)

In [40]:
q_traj, energy, _ = grasp_transfer_opt.run_adam(
  sample_grasp_q.squeeze(0), running_name="test"
)

In [ ]:
print(q_traj.shape)
best_q = q_traj[21, -1]
print(best_q.shape)

In [ ]:
best_q.shape[0]

In [ ]:
target_model.dynamic_joints_q_lower.shape

In [ ]:
midjoints = torch.tensor(target_model.dynamic_joints_q_mid)
print(midjoints)

In [ ]:
target_model.dynamic_joints_q_upper[0]

In [ ]:
target_model.dynamic_joints_q_lower[0]

In [58]:
if best_q.shape[0] != 9 + len(target_model.dynamic_joints):
  # We optimized only for pose, so need to provide dummy joints
  best_q = torch.cat((best_q, midjoints), dim=0)

In [ ]:
print(best_q)

## Visualize results

In [ ]:
print("Plotting TARGET and SOURCE together...")

vis_data = source_model.get_plotly_data(q=sample_grasp_q, color='red')
vis_data += target_model.get_plotly_data(q=best_q.unsqueeze(0).float().to(device), color='green')
fig = go.Figure(data=vis_data)
fig.show()
# fig.write_html("../logs_viz/gtransfer_test.html")


In [ ]:
base_pose = torch.zeros(9)
# Identity rotation in 6d rot representation is: (1,0,0,0,1,0)
base_pose[3] = 1
base_pose[7] = 1 
print("Base Pose:", base_pose)

base_q = torch.cat((base_pose, midjoints), dim=0)
print("Base Q:", base_q)

In [ ]:
vis_data_target_base = target_model.get_plotly_data(q=base_q.unsqueeze(0).float().to(device), color='green')
print(len(vis_data_target_base))


In [75]:
def combine_mesh3d_list(mesh3d_list):
    """
    Combine a list of Plotly mesh3d objects into a single Trimesh object.

    Parameters:
        mesh3d_list: list of dict-like objects, where each represents a Plotly mesh3d object.

    Returns:
        A combined Trimesh object.
    """
    all_vertices = []
    all_faces = []
    vertex_offset = 0

    for mesh3d in mesh3d_list:
        # Extract vertices and faces
        vertices = np.array([mesh3d['x'], mesh3d['y'], mesh3d['z']]).T
        faces = np.array([mesh3d['i'], mesh3d['j'], mesh3d['k']]).T

        # Offset face indices by the current number of vertices
        faces += vertex_offset

        # Append to the lists
        all_vertices.append(vertices)
        all_faces.append(faces)

        # Update the vertex offset
        vertex_offset += vertices.shape[0]

    # Combine all vertices and faces
    combined_vertices = np.vstack(all_vertices)
    combined_faces = np.vstack(all_faces)

    # Create the combined Trimesh object
    combined_mesh = trimesh.Trimesh(vertices=combined_vertices, faces=combined_faces)

    return combined_mesh


In [77]:
comb_mesh = combine_mesh3d_list(vis_data_target_base)

In [80]:
# comb_mesh.show()

In [ ]:
comb_mesh.export("fetch_gripper_base_pose.obj")